# OneVoice V2 — streaming semantics và reliability

Notebook này chỉ điều phối Colab/Drive. Runtime thật nằm trong `src/pipeline.py`: WAV được phát lại thành frame 32 ms, đi qua denoiser/VAD/ASR rolling/semantic commit/MT/TTS và ghi trace cùng latency. Không phát ra loa trong chế độ replay.

Mặc định dùng safety WAV đã có checksum để smoke test nhanh; đổi `DIRECTION` và `INPUT_FILE` nếu muốn chạy một WAV khác. Báo cáo được ghi trên Drive và có thể chạy lại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shutil, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'model_cache/huggingface')
# Avoid a slow reinstall on every Run all. A fresh subprocess validates the complete stack first.
stack_probe = [sys.executable, '-c', 'import numpy, transformers, tokenizers, sentencepiece; from transformers import T5ForConditionalGeneration; import sherpa_onnx, funasr_onnx, pyttsx3; print("stack-ok", numpy.__version__, transformers.__version__, tokenizers.__version__, sentencepiece.__version__)']
probe = subprocess.run(stack_probe, text=True, capture_output=True)
if probe.returncode:
    print('Installing/repairing Colab dependencies (first run can take several minutes)...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'soundfile', 'PyYAML', 'sherpa-onnx', 'funasr-onnx', 'pyttsx3', 'sacremoses'], check=True)
    # Remove mixed package files; pip force-reinstall alone can leave lazy-import modules from an older release.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'transformers', 'tokenizers', 'sentencepiece'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir', '--upgrade', 'numpy==2.2.6', 'transformers==4.57.1', 'tokenizers==0.22.1', 'sentencepiece==0.2.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False)
else:
    print('Colab dependencies already valid; skipping reinstall.', flush=True)
if shutil.which('espeak-ng') is None and shutil.which('espeak') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'espeak-ng'], check=True)
backend_check = subprocess.run(stack_probe, text=True, capture_output=True)
if backend_check.returncode:
    print(backend_check.stdout, end=''); print(backend_check.stderr, end='')
    raise RuntimeError('Required streaming backends are unavailable; restart the Colab runtime and run this setup cell again.')
print(backend_check.stdout, end='')
CONFIG = DRIVE_ROOT / 'configs/runtime_demo_local.yaml'
DIRECTION = 'vi2en'  # 'vi2en' hoặc 'en2vi'
REPORT_DIR = DRIVE_ROOT / 'reports/streaming_v2_smoke' / DIRECTION
DEFAULT_INPUT = DRIVE_ROOT / 'artifacts/safety_audio_v1' / (
    'SAFE2_0001_en2vi.wav' if DIRECTION == 'vi2en' else 'SAFE2_0001_vi2en.wav'
)
INPUT_FILE = DEFAULT_INPUT  # thay bằng Path('/content/drive/MyDrive/.../file.wav') nếu cần
if not CONFIG.is_file(): raise FileNotFoundError(f'Missing runtime config: {CONFIG}')
if not INPUT_FILE.is_file(): raise FileNotFoundError(f'Missing input WAV: {INPUT_FILE}')
print('Source:', REPO)
print('Direction:', DIRECTION, '| Input:', INPUT_FILE, '| Reports:', REPORT_DIR)


In [ ]:
command = [
    sys.executable, 'src/pipeline.py',
    '--config', str(CONFIG), '--direction', DIRECTION,
    '--profile', 'development', '--offline',
    '--stream-file', str(INPUT_FILE), '--report-dir', str(REPORT_DIR),
]
print('> ' + ' '.join(map(str, command)), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
code = process.wait()
if code:
    raise RuntimeError(f'Streaming smoke failed with exit code {code}; inspect the full log above and stream_result.json on Drive.')
# Accept either a pathlib.Path or a string override from a previous Colab cell.
REPORT_DIR = Path(REPORT_DIR)
result_path = REPORT_DIR / 'stream_result.json'
result = json.loads(result_path.read_text(encoding='utf-8'))
print(json.dumps({k: result[k] for k in ('direction', 'frame_samples', 'frame_ms', 'frames_submitted', 'commits', 'dropped_audio_frames', 'fatal_error')}, ensure_ascii=False, indent=2))


In [ ]:
REPORT_DIR = Path(REPORT_DIR)
result = json.loads((REPORT_DIR / 'stream_result.json').read_text(encoding='utf-8'))
latency = json.loads((REPORT_DIR / 'latency_summary.json').read_text(encoding='utf-8')) if (REPORT_DIR / 'latency_summary.json').is_file() else {}
print('Commits:', result['commits'], '| commit IDs:', result['commit_ids'])
print('Latency summary:', json.dumps(latency, ensure_ascii=False, indent=2))
print('Hypothesis events:', len(result['hypothesis_trace']), '| chunks:', len(result['chunks']))
print('Report directory:', REPORT_DIR)


## Fixed normal + safety streaming gate

Cell này chọn cố định WAV `test` sạch cho normal path và WAV safety đã checksum cho safety path, ở cả hai chiều. Mỗi case chạy qua đúng worker graph 32 ms, kiểm tra commit không trùng/đảo thứ tự, worker error, dropped frame, route và audio chunk. `--resume` giữ kết quả hợp lệ trên Drive, nên có thể chạy lại sau khi Colab ngắt.

Mặc định là 1 normal + 1 safety mỗi chiều để xác nhận integration nhanh. Tăng `CASES_PER_ROUTE` sau khi smoke pass.

In [ ]:
VI_MANIFEST = Path('/content/drive/MyDrive/onevoice_audio_v1/manifest.jsonl')
EN_MANIFEST = Path('/content/drive/MyDrive/onevoice_audio_v2_1/manifest.jsonl')
SUITE_REPORT_DIR = DRIVE_ROOT / 'reports/streaming_v2_fixed_suite_v1'
CASES_PER_ROUTE = 1
command = [
    sys.executable, 'scripts/run_streaming_e2e.py',
    '--config', str(CONFIG),
    '--vi-manifest', str(VI_MANIFEST),
    '--en-manifest', str(EN_MANIFEST),
    '--output-dir', str(SUITE_REPORT_DIR),
    '--cases-per-route', str(CASES_PER_ROUTE),
    '--profile', 'development', '--offline', '--resume',
]
print('> ' + ' '.join(map(str, command)), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError(f'Fixed streaming suite failed; inspect {SUITE_REPORT_DIR / "summary.json"}.')
summary = json.loads((SUITE_REPORT_DIR / 'summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 30-minute real-time soak (P2-09)

Chạy sau khi fixed suite pass. Runner luân phiên 1 normal và 1 safety case ở cả hai chiều trong thời gian thực, ghi `events.jsonl` và `soak_state.json` sau từng turn. Nếu Colab ngắt hoặc đổi account, chạy đúng cell này lần nữa: `--resume` sẽ cộng dồn thời lượng đã chạy, không bắt đầu lại từ 0.

Không đổi `DURATION_MINUTES` giữa các lần resume. Nếu short soak 30 phút pass, chỉ khi đó mới chạy lại với output directory mới và `120` phút.

In [ ]:
SOAK_REPORT_DIR = DRIVE_ROOT / 'reports/streaming_v2_soak_30m_v1'
DURATION_MINUTES = 30
command = [
    sys.executable, 'scripts/run_streaming_soak.py',
    '--config', str(CONFIG),
    '--vi-manifest', str(VI_MANIFEST),
    '--en-manifest', str(EN_MANIFEST),
    '--output-dir', str(SOAK_REPORT_DIR),
    '--duration-minutes', str(DURATION_MINUTES),
    '--profile', 'development', '--offline', '--realtime', '--resume',
]
print('> ' + ' '.join(map(str, command)), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError(f'Soak stopped or failed; inspect {SOAK_REPORT_DIR / "summary.json"} and events.jsonl.')
summary = json.loads((SOAK_REPORT_DIR / 'summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
